# Project Fertility PSP

난임 환자 대상 임신 성공 여부 예측 프로젝트 공식 분석 노트북

- PSP: Pregnancy Success Prediction
- 목표: 난임 치료 관련 데이터를 활용하여 임신 성공 여부를 예측하는 분류 모델 개발
- 용도: 팀 논의, 분석 방향 정리, 모델링 전략 공유


## 00. 전체 분석 방향

본 프로젝트는 단순히 모델 하나를 학습하는 방식이 아니라, 데이터 탐색 결과를 전처리와 파생변수 설계로 연결하고, 여러 모델을 비교한 뒤 최종 제출 전략을 결정하는 흐름으로 진행한다.

전체 단계는 다음과 같다.

1. EDA: 데이터 구조, 결측치, 이상치, 변수 관계 파악
2. Preprocessing: 결측/이상치 처리, 인코딩, 스케일링, 데이터 분리
3. Modeling: 분류 모델 적용, 파생변수 추가, K-Fold, 모델 비교
4. Evaluation: ROC-AUC 중심의 성능 평가
5. Submission: 제출 파일 생성 및 리더보드 확인


## 01. EDA

EDA는 단순히 표와 그래프를 확인하는 단계가 아니라, 이후 모델링에 사용할 변수를 설계하기 위한 단계이다.


### 01-1. Target Distribution

- `임신 성공 여부`의 0/1 비율을 확인한다.
- 성공 클래스가 실패 클래스보다 적은 불균형 데이터인지 확인한다.
- 불균형이 존재하므로 Accuracy만으로 모델을 평가하지 않는다.
- 핵심 평가지표는 ROC-AUC로 설정한다.


### 01-2. Missing Values

- 결측치가 많은 변수를 확인한다.
- 결측치가 단순 누락인지, 특정 시술 유형과 연결된 구조적 결측인지 구분한다.
- 예를 들어 배아 관련 변수의 결측은 DI/IVF 시술 구조 차이에서 발생할 수 있다.
- 결측 여부 자체가 예측에 의미를 가질 수 있으므로, 무조건 삭제하지 않는다.


### 01-3. Outliers

- 수치형 변수에서 비정상적으로 큰 값이나 작은 값을 확인한다.
- IQR 기준 또는 도메인 기준으로 이상치 후보를 탐색한다.
- 단, 의료/시술 데이터에서는 극단값이 실제 기록일 수 있으므로 삭제보다 보존 또는 구간화가 적절할 수 있다.


### 01-4. Feature Relationships

임신 성공 여부와 관련이 큰 변수 후보를 확인한다.

- 시술 당시 나이
- 시술 유형
- 특정 시술 유형
- 배아 이식 경과일
- 단일 배아 이식 여부
- 배아 생성 주요 이유
- 총 시술 횟수
- 난자 출처
- 정자 출처
- 신선/동결 배아 사용 여부


### 01-5. EDA에서 도출한 모델링 아이디어

- 나이는 순서가 있는 범주형 변수이므로 순서형 숫자로 변환할 수 있다.
- 특정 시술 유형은 `ICSI`, `IVF`, `BLASTOCYST`, `AH` 등 토큰 단위로 분해할 수 있다.
- 배아 이식 경과일은 5일 여부, 4-5일 구간 여부 등으로 파생변수를 만들 수 있다.
- 난자 출처와 나이를 조합하면 중요한 상호작용 변수가 될 수 있다.
- 결측 개수, 결측 비율, 결측 여부 자체를 파생변수로 사용할 수 있다.


## 02. Preprocessing

전처리는 모델이 데이터를 안정적으로 학습할 수 있도록 정리하는 단계이다. 본 프로젝트에서는 결측치와 범주형 변수가 많기 때문에 전처리 방식이 성능에 큰 영향을 줄 수 있다.


### 02-1. Missing Value Handling

- 수치형 변수는 중앙값 대체를 기본으로 고려한다.
- 범주형 변수는 `Unknown` 또는 `__MISSING__` 같은 별도 범주로 처리한다.
- 결측률이 높은 변수는 삭제 여부를 바로 결정하지 않고, 결측 여부 변수를 함께 만든다.
- 시술 유형에 따라 발생하는 구조적 결측은 중요한 정보로 활용한다.


### 02-2. Outlier Handling

- 이상치 후보는 먼저 분포를 확인한다.
- 모델 성능에 악영향을 줄 정도로 극단적인 경우에만 clipping 또는 구간화를 고려한다.
- 트리 기반 모델은 이상치에 비교적 강하므로, 모든 이상치를 기계적으로 제거하지 않는다.


### 02-3. Scaling

- Logistic Regression처럼 거리나 계수 크기에 민감한 모델은 스케일링이 필요하다.
- Random Forest, CatBoost, LightGBM, XGBoost 같은 트리 기반 모델은 스케일링 영향이 상대적으로 작다.
- 따라서 모델 종류에 따라 스케일링 필요 여부를 다르게 판단한다.


### 02-4. Encoding

- 기본 모델에서는 One-Hot Encoding을 사용할 수 있다.
- CatBoost는 범주형 변수를 직접 처리할 수 있으므로 별도 One-Hot Encoding이 필수는 아니다.
- LightGBM/XGBoost에서는 ordinal encoding, frequency encoding 등 다양한 인코딩을 검토할 수 있다.
- 인코딩 과정에서 validation 정보가 train에 섞이지 않도록 데이터 누수를 주의한다.


### 02-5. Feature Engineering

EDA 결과를 바탕으로 다음 파생변수를 우선 검토한다.

- 나이 순서형 변수
- 38세 이상, 40세 이상, 43세 이상 여부
- BLASTOCYST 포함 여부
- AH 포함 여부
- ICSI 포함 여부
- 배아 이식 경과일 5일 여부
- 배아 관련 결측 개수
- 생성 배아 대비 이식 배아 비율
- 난자 기증 여부
- 나이와 난자 출처 조합
- 시술 유형과 특정 시술 유형 조합


### 02-6. Train/Valid Split

- 학습 데이터와 검증 데이터를 분리한다.
- 타깃 비율을 유지하기 위해 Stratified Split을 사용한다.
- 단일 split 결과만 믿기보다, 최종 단계에서는 K-Fold 검증을 적용한다.


## 03. Modeling

모델링 단계에서는 여러 분류 모델을 비교하고, 성능이 좋은 모델을 중심으로 고도화한다.


### 03-1. Baseline Models

- Logistic Regression: 기준선 모델로 사용한다.
- Random Forest: 비선형 관계를 반영할 수 있는 기본 트리 모델이다.
- Baseline 모델은 이후 고도화 모델의 성능을 판단하기 위한 비교 기준이다.


### 03-2. Main Candidate Models

- CatBoost: 범주형 변수가 많고 결측 구조가 있는 데이터에 적합하다.
- LightGBM: 빠르고 성능이 좋은 Gradient Boosting 모델이다.
- XGBoost: 안정적인 Gradient Boosting 모델이며 앙상블 다양성을 높인다.
- 각 모델은 서로 다른 방식으로 패턴을 학습하므로 함께 비교한다.


### 03-3. Hyperparameter Tuning

모델 성능 향상을 위해 다음 설정값을 조정한다.

- learning rate
- depth 또는 max_depth
- regularization 계수
- iterations 또는 n_estimators
- min_child_samples, min_child_weight
- subsample, colsample_bytree
- early stopping 기준


### 03-4. K-Fold Validation

- 데이터를 K개 fold로 나누어 학습과 검증을 반복한다.
- 각 행은 한 번씩 validation 데이터가 된다.
- 모든 validation 예측값을 모아 OOF 예측값을 만든다.
- OOF ROC-AUC는 모델의 일반화 성능을 판단하는 중요한 내부 기준이다.


### 03-5. Ensemble / Stacking

- Ensemble: 여러 모델의 예측값을 평균 또는 가중 평균하여 최종 예측을 만든다.
- Stacking: 여러 모델의 예측값을 다시 새로운 모델의 입력으로 사용한다.
- 본 프로젝트에서는 우선 OOF 기반 가중 평균 앙상블을 검토한다.
- 모델별 OOF ROC-AUC를 기준으로 가중치를 조정할 수 있다.


## 04. Evaluation

평가 단계에서는 단순 정확도보다 예측 확률의 순위와 품질을 중심으로 본다.


### 04-1. Core Metric: ROC-AUC

- ROC-AUC는 모델이 성공과 실패를 얼마나 잘 구분하는지 보는 지표이다.
- 1에 가까울수록 좋고, 0.5는 무작위 예측 수준이다.
- 대회 리더보드 기준과 맞추기 위해 ROC-AUC를 핵심 지표로 사용한다.


### 04-2. Supporting Metrics

- LogLoss: 예측 확률이 실제 정답과 얼마나 가까운지 평가한다.
- Accuracy: 전체 예측 중 맞힌 비율이다.
- Precision: 성공이라고 예측한 것 중 실제 성공한 비율이다.
- Recall: 실제 성공 사례 중 모델이 성공으로 찾아낸 비율이다.
- F1-score: Precision과 Recall의 균형을 평가한다.


### 04-3. Model Comparison

- 모든 모델의 validation 또는 OOF ROC-AUC를 비교한다.
- 단일 모델 성능과 앙상블 성능을 함께 비교한다.
- LogLoss가 낮은 모델은 확률 품질 측면에서 참고한다.
- 리더보드 제출 전에는 OOF 점수와 제출 점수의 차이를 기록한다.


## 05. Submission

제출 단계에서는 예측 성능뿐 아니라 제출 파일 형식 검증이 중요하다.


### 05-1. Submission File

- `sample_submission.csv`를 기준으로 제출 파일을 만든다.
- 제출 파일은 `ID`, `probability` 두 컬럼으로 구성한다.
- `probability`에는 임신 성공 확률 예측값을 넣는다.
- 값은 0과 1 사이의 확률이어야 한다.


### 05-2. Submission Validation

제출 전 반드시 다음 항목을 확인한다.

- 행 개수가 test 데이터와 같은가
- ID 순서가 test 데이터와 같은가
- 컬럼명이 대회 제출 형식과 같은가
- probability에 결측치가 없는가
- probability 값이 0과 1 사이인가
- 불필요한 컬럼이 추가되지 않았는가


### 05-3. Leaderboard Check

- 제출 후 리더보드 점수를 기록한다.
- 내부 OOF ROC-AUC와 리더보드 점수 차이를 비교한다.
- 점수 변화가 있는 경우 어떤 전처리, 모델, 앙상블 조합을 사용했는지 함께 기록한다.
- 최종 발표에서는 모델 개선 과정과 점수 변화 근거를 함께 설명한다.


## 팀 논의 안건

다음 회의에서 우선 논의할 항목은 다음과 같다.

1. 구조적 결측을 어떤 방식으로 처리할 것인가
2. 어떤 파생변수를 우선 추가할 것인가
3. CatBoost 단일 모델과 LightGBM/XGBoost 앙상블 중 어떤 전략을 먼저 실험할 것인가
4. K-Fold를 3-Fold로 할지 5-Fold로 할지 결정
5. 리더보드 제출 파일과 실험 기록을 누가 관리할 것인가
